# 07 · X-ray fluorescence of brass (Práctica 6)

EPN Práctica 6 on the 554 800 with the Si-PIN energy detector LD 559 938: the sample sits at 45° to the beam and the
detector at 90°. The MCA (512 channels, gain −2.5, 180 s, 35 kV, 1 mA) records Cu, Zn, Pb, leaded brass CuZn39Pb3 and
a coin. Two Cu/Pb lines calibrate the channels; three equal-width Gaussians separate Cu Kα, Zn Kα and Cu Kβ; the line
areas against the pure elements give the composition to first order. The model lives in `braggsim.fluorescence`,
`braggsim.detector` and `braggsim.analysis`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from braggsim import fluorescence as fl
from braggsim.analysis import composition, gauss_fit_lines, two_point_calibration
from braggsim.detector import kev_per_channel, pin_fwhm_keV, sample_counts

plt.rcParams.update(
    {
        "axes.grid": True,
        "grid.color": "#e5e5e5",
        "lines.linewidth": 1.2,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
DATA, MODEL, ALT = "#1a1a19", "#2a78d6", "#eb6834"
rng = np.random.default_rng(2026)
T_S, U_KV, I_MA = 180.0, 35.0, 1.0
NAMES = ["Cu", "Zn", "Pb", "CuZn39Pb3", "centavo2003"]
spec = {n: fl.mca_spectrum(n, U_KV, I_MA, T_S) for n in NAMES}  # (counts, live_s, E_keV)
draw = {n: sample_counts(c, 1.0, rng) for n, (c, _, _) in spec.items()}  # one Poisson draw
pd.DataFrame(
    {n: {"live fraction": live / T_S, "counts": c.sum()} for n, (c, live, _) in spec.items()}
).T

## MCA spectra

Expected counts (line) and one Poisson draw (dots). Cu Kβ sits on Zn Kα in the brass; the peak at ~17 keV is the Mo Kα Compton line, ~17.5 keV the Rayleigh line.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 5), sharex=True)
for j, n in enumerate(NAMES):
    c, _, e = spec[n]
    for i, log in enumerate((False, True)):
        ax = axes[i, j]
        ax.plot(e, draw[n], ".", ms=2, color=DATA)
        ax.plot(e, c, color=MODEL)
        if log:
            ax.set_yscale("log")
            ax.set_ylim(0.5, None)
        if i == 0:
            ax.set_title(n)
        ax.set_xlim(0, 25)
        ax.set_xlabel("E (keV, true scale)")
plt.tight_layout()

## Two-point calibration

The guide calibrates on Cu Kα (8.04 keV) and Pb Lα (10.56 keV): read the channel of each maximum, then E = k·ch + E₀.

In [ ]:
def peak_ch(n, lo, hi):
    return lo + int(np.argmax(draw[n][lo:hi]))


ch_cu, ch_pb = peak_ch("Cu", 80, 130), peak_ch("Pb", 115, 150)
k, e0 = two_point_calibration(ch_cu, 8.04, ch_pb, 10.56)
print(f"Cu Ka ch {ch_cu}, Pb La ch {ch_pb}: {k:.4f} keV/channel, offset {e0:+.3f} keV")
print(f"true: {kev_per_channel():.4f} keV/channel, offset 0")

## Fitting the brass blend

Two fits of the same Poisson draw, compared with the true expected line counts at the detector
(`detected_line_counts`).

- **LD's method**: three equal-width Gaussians (Cu Kα 8.04, Zn Kα 8.64, Cu Kβ 8.91 keV) in 7.6–9.2 keV. The window must stop
  there: above it Zn Kβ (9.57 keV) leaks into the last Gaussian.
- **Key fit**: a fourth Gaussian at 9.57 keV and a window of 7.5–10.0 keV, which removes that bias.

Centres are free within ±0.1 keV, linear background, Poisson weights.

One Poisson draw scatters: the Zn Kα / Cu Kβ split is the weak point of the blend (errors of a few percent, anticorrelated), so a single draw can sit two σ from the truth. Repeating the draw with a different seed shows the scatter; the fit itself is unbiased.


In [ ]:
c, live, e_true = spec["CuZn39Pb3"]
E = k * np.arange(512) + e0  # the students' axis
y = draw["CuZn39Pb3"]
fit_ld = gauss_fit_lines(E, y, [8.04, 8.64, 8.91], (7.6, 9.2))  # LD's method
fit = gauss_fit_lines(E, y, [8.04, 8.64, 8.91, 9.57], (7.5, 10.0))  # key fit
true = fl.detected_line_counts("CuZn39Pb3", U_KV, I_MA, T_S)
names = ["Cu Kα", "Zn Kα", "Cu Kβ"]
pd.DataFrame(
    {
        "LD fit": fit_ld.area,
        "key fit": fit.area[:3],
        "key ± (1σ)": fit.area_err[:3],
        "true expected": [true[x] for x in names],
    },
    index=names,
).round(1)

In [ ]:
m = (E > 7.4) & (E < 10.1)
plt.figure(figsize=(6, 3.5))
plt.plot(E[m], y[m], ".", color=DATA)
plt.xlabel("E (keV)")
plt.ylabel("counts / channel")
xs = np.linspace(7.4, 10.1, 400)
parts = fit.evaluate(xs)
for g in parts:
    plt.plot(xs, g, color=ALT, lw=0.8)
plt.plot(xs, parts.sum(axis=0), color=MODEL)

## Composition

Standards: pure Cu (Kα), pure Zn (Kα), pure Pb (Lα), fitted the same way. Pb Lα is a weak line on the Zn Kβ tail, so it is fitted
in 10.1–11.0 keV with the width fixed to the detector's (`pin_fwhm_keV`), as a free width and the sloping background are degenerate there.
Plain: w ∝ A/A_pure. ρ-weighted (LD's method): w ∝ ρ·A/A_pure, since A/A_pure ≈ n/n_pure = w ρ_alloy/ρ_i. LD's published result for
CuZn39Pb3 is 61.6 / 35.6 / 2.9 %. Errors propagate the fit covariance (Cu Kα and Zn Kα of one blend are positively correlated) and the standards' errors.

In [ ]:
sig_pb = float(pin_fwhm_keV(10.55)) / (2 * np.sqrt(2 * np.log(2)))


def area(n, centers, win, **kw):
    return gauss_fit_lines(E, draw[n], centers, win, **kw)


pb_alloy = area("CuZn39Pb3", [10.55], (10.1, 11.0), sigma_keV=sig_pb)
fits_pure = {
    "Cu": area("Cu", [8.04], (7.4, 8.5)),
    "Zn": area("Zn", [8.63], (8.0, 9.2)),
    "Pb": area("Pb", [10.55], (10.1, 11.0), sigma_keV=sig_pb),
}
a_alloy = {"Cu": fit.area[0], "Zn": fit.area[1], "Pb": pb_alloy.area[0]}
a_pure = {k_: f.area[0] for k_, f in fits_pure.items()}
cov = np.zeros((3, 3))
cov[:2, :2] = fit.area_cov[:2, :2]
cov[2, 2] = pb_alloy.area_cov[0, 0]
err_pure = {k_: f.area_err[0] for k_, f in fits_pure.items()}
rho = {"Cu": 8.96, "Zn": 7.10, "Pb": 11.34}
plain = composition(a_alloy, a_pure, alloy_cov=cov, areas_pure_err=err_pure)
weighted = composition(a_alloy, a_pure, rho, alloy_cov=cov, areas_pure_err=err_pure)
tab = pd.DataFrame(
    {
        "plain": {k_: v[0] for k_, v in plain.items()},
        "rho-weighted": {k_: v[0] for k_, v in weighted.items()},
        "± (1σ)": {k_: v[1] for k_, v in weighted.items()},
        "LD": {"Cu": 61.6, "Zn": 35.6, "Pb": 2.9},
        "nominal": {"Cu": 58, "Zn": 39, "Pb": 3},
    }
)
tab.round(1)

## Why the first-order method is biased

The alloy is not a mixture of pure standards. With the true line counts, A/A_pure is 0.543 for Cu, 0.396 for Zn and 0.026 for Pb:
plain normalisation gives about 56 / 41 / 2.7 %, so Cu reads **low**. In the alloy Zn, and above all Pb, absorb Cu Kα, and that beats
the extra Cu Kα from secondary fluorescence (Zn Kβ and Pb L above the Cu K edge, 8.98 keV; ≈ 4 % in this model).

LD's 61.6 / 35.6 is reproduced because LD's method weights by the density ρ_i (×8.96 for Cu, ×7.10 for Zn). That is the right logic for thin
foils of equal size; for an infinitely thick sample the density cancels (A/A_pure ≈ w_i μ*_pure/μ*_alloy), so the weighting itself is a bias
that happens to push Cu up. The model gets this from the same recipe, not from tuning. It gives Pb ≈ 3.7 against LD's 2.9: the Pb Lα peak
sits on the tail of Zn Kβ, and LD's Pb agreement is not reproduced to better than about one percentage point.

## The coin

The Ecuador 1 centavo 2003 is a Cu plating (estimated 18 µm) on a steel core. Cu Kα travels ≈ 21 µm in Cu, so the core's Fe Kα comes through weakened, and how much depends on the plating thickness.

In [ ]:
t_um = np.array([5, 10, 15, 18, 25, 40])
fe = [
    next(x for x in fl.line_rates(fl.coin(t), U_KV, I_MA) if x.label.startswith("Fe Kα")).rate_per_s
    for t in t_um
]
plt.figure(figsize=(5, 3))
plt.plot(t_um, fe, "o-", color=MODEL)
plt.yscale("log")
plt.xlabel("Cu plating (µm)")
plt.ylabel("Fe Kα at the window (photons/s)")